In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



In [7]:
#load the dataset
df= pd.read_excel("indian_eco_system.xlsx")
df

,order_id,order_date,state,zone,category,brand_type,customer_gender,customer_age,base_price,discount_percent,final_price,units_sold,revenue,sales_event,competition_intensity,inventory_pressure
0,ORD0000001,2023-01-31,West Bengal,East,Fashion,Mass,Male,28,1810.89,65.00,633.81,29,18380.49,Normal,Medium,High
1,ORD0000002,2023-01-31,Gujarat,West,Sports & Fitness,Mass,Male,19,5678.15,50.83,2791.95,67,187060.65,Normal,Medium,High
2,ORD0000003,2023-01-31,Delhi NCR,North,Grocery Essentials,Mass,Male,25,169.98,35.26,110.05,57,6272.85,Normal,Medium,Low
3,ORD0000004,2023-01-31,Madhya Pradesh,Central,Footwear,Mass,Female,30,3244.91,61.94,1235.01,55,67925.55,Normal,High,High
4,ORD0000005,2023-01-31,Haryana,North,Fashion,Premium,Female,33,697.44,48.27,360.79,29,10462.91,Festival,Medium,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30595,ORD0030596,2025-12-31,Uttar Pradesh,North,Premium Lifestyle,Mass,Male,54,12463.98,38.99,7604.27,11,83646.97,Normal,Medium,Low
30596,ORD0030597,2025-12-31,Haryana,North,Footwear,Premium,Male,18,3974.58,39.35,2410.58,20,48211.60,Festival,Low,Low
30597,ORD0030598,2025-12-31,Odisha,East,Footwear,Mass,Female,58,3020.42,65.00,1057.15,59,62371.85,Festival,High,Low
30598,ORD0030599,2025-12-31,West Bengal,East,Sports & Fitness,Mass,Female,23,1212.05,12.35,1062.36,8,8498.88,Normal,Medium,Low


In [8]:
#Select the required target numerical columns
target_cols = ['base_price', 'discount_percent', 'final_price', 'units_sold', 'revenue']

#Compute Mean, Median (50%), Min, and Max
summary_metrics = df[target_cols].agg(['mean', 'median', 'min', 'max']).T

#Rename columns for clear reporting
summary_metrics.columns = ['Mean', 'Median (50%)', 'Min', 'Max']

#Display formatted summary dataframe
print(summary_metrics.round(2))

                      Mean  Median (50%)     Min        Max
base_price         3898.64       2975.02  100.03   14999.17
discount_percent     38.11         37.29    0.01      65.00
final_price        2415.02       1728.60   35.01   14682.28
units_sold           31.71         27.00    2.00     161.00
revenue           71062.43      42138.51  247.80  906141.04


In [9]:
# 1. Zonal Demand Breakdown
zonal_summary = df.groupby('zone').agg(
    total_revenue=('revenue', 'sum'),
    total_units=('units_sold', 'sum'),
    avg_discount=('discount_percent', 'mean')
).sort_values(by='total_revenue', ascending=False)
pd.options.display.float_format = '{:.2f}'.format
print("=== 1. ZONAL DEMAND SUMMARY ===")
print(zonal_summary.round(2))
print("\n")

=== 1. ZONAL DEMAND SUMMARY ===
         total_revenue  total_units  avg_discount
zone                                             
South     689433867.25       303941         38.01
North     636512340.45       285754         38.18
West      357659151.06       159156         37.46
Central   246574584.90       112886         38.31
East      244330423.33       108451         38.66




In [10]:
# 2. Category Performance Breakdown
category_summary = df.groupby('category').agg(
    total_revenue=('revenue', 'sum'),
    total_units=('units_sold', 'sum'),
    avg_price=('final_price', 'mean')
).sort_values(by='total_revenue', ascending=False)

print("=== 2. CATEGORY PERFORMANCE SUMMARY ===")
print(category_summary.round(2))
print("\n")

=== 2. CATEGORY PERFORMANCE SUMMARY ===
                        total_revenue  total_units  avg_price
category                                                     
Premium Lifestyle        648096910.69       119114    5872.79
Electronics              520517491.14       120702    4654.82
Sports & Fitness         283366646.04       122809    2491.24
Home & Living            236678299.13       119991    2122.18
Footwear                 195768282.63       120246    1738.04
Fashion                  157875250.20       123294    1391.45
Beauty & Personal Care   101623204.40       125456     873.61
Grocery Essentials        30584282.76       118576     276.62




In [11]:
# 3. Festival vs Normal Sales Surge
event_summary = df.groupby('sales_event').agg(
    avg_units_per_order=('units_sold', 'mean'),
    avg_discount=('discount_percent', 'mean'),
    total_revenue=('revenue', 'sum')
)

print("=== 3. SALES EVENT SURGE ANALYSIS ===")
print(event_summary.round(2))

=== 3. SALES EVENT SURGE ANALYSIS ===
             avg_units_per_order  avg_discount  total_revenue
sales_event                                                  
Festival                   48.05         44.86   760689249.54
Normal                     26.20         35.84  1413821117.45


In [12]:
# Clean formatting (removes scientific notation e+08)
pd.options.display.float_format = '{:,.2f}'.format

# ---------------------------------------------------------
# 1. ABC Analysis (Revenue Prioritization)
# ---------------------------------------------------------
cat_abc = df.groupby('category').agg(
    total_revenue=('revenue', 'sum')
).sort_values(by='total_revenue', ascending=False).reset_index()

cat_abc['cum_revenue'] = cat_abc['total_revenue'].cumsum()
cat_abc['cum_perc'] = (cat_abc['cum_revenue'] / cat_abc['total_revenue'].sum()) * 100

def assign_abc(perc):
    if perc <= 70:
        return 'A'
    elif perc <= 90:
        return 'B'
    else:
        return 'C'

cat_abc['ABC_Class'] = cat_abc['cum_perc'].apply(assign_abc)

# ---------------------------------------------------------
# 2. XYZ Analysis (Demand Volatility via CV)
# ---------------------------------------------------------
df['order_date'] = pd.to_datetime(df['order_date'], format='%d/%m/%Y')
monthly_cat = df.groupby(['category', df['order_date'].dt.to_period('M')])['units_sold'].sum().reset_index()

xyz_stats = monthly_cat.groupby('category')['units_sold'].agg(['mean', 'std']).reset_index()
xyz_stats['CV'] = xyz_stats['std'] / xyz_stats['mean']

def assign_xyz(cv):
    if cv <= 0.10:
        return 'X'
    elif cv <= 0.25:
        return 'Y'
    else:
        return 'Z'

xyz_stats['XYZ_Class'] = xyz_stats['CV'].apply(assign_xyz)

# Merge ABC & XYZ into Combined Matrix
abc_xyz = pd.merge(cat_abc, xyz_stats[['category', 'CV', 'XYZ_Class']], on='category')
abc_xyz['ABC_XYZ_Matrix'] = abc_xyz['ABC_Class'] + abc_xyz['XYZ_Class']

print("=== 1. ABC/XYZ CLASSIFICATION MATRIX ===")
print(abc_xyz[['category', 'total_revenue', 'cum_perc', 'ABC_Class', 'CV', 'XYZ_Class', 'ABC_XYZ_Matrix']])
print("\n")

# ---------------------------------------------------------
# 3. Safety Stock (SS) & Reorder Point (ROP) Calculations
# Assumptions: Service Level = 95% (Z = 1.65), Lead Time = 7 Days (L = 7)
# ---------------------------------------------------------
L = 7
Z = 1.65

zone_cat = df.groupby(['zone', 'category']).agg(
    total_units=('units_sold', 'sum'),
    std_units=('units_sold', 'std')
).reset_index()

# Daily demand across 36 months (~1,095 days)7
zone_cat['daily_demand'] = zone_cat['total_units'] / (36 * 30.4)
zone_cat['safety_stock'] = (Z * zone_cat['std_units'] * np.sqrt(L)).round(0)
zone_cat['reorder_point'] = ((zone_cat['daily_demand'] * L) + zone_cat['safety_stock']).round(0)

print("=== 2. SAFETY STOCK & ROP (TOP REGIONAL NODES) ===")
print(zone_cat.sort_values(by='total_units', ascending=False).head(10)[['zone', 'category', 'daily_demand', 'safety_stock', 'reorder_point']])

=== 1. ABC/XYZ CLASSIFICATION MATRIX ===
                 category  total_revenue  cum_perc ABC_Class   CV XYZ_Class  \
0       Premium Lifestyle 648,096,910.69     29.80         A 0.10         Y   
1             Electronics 520,517,491.14     53.74         A 0.10         Y   
2        Sports & Fitness 283,366,646.04     66.77         A 0.09         X   
3           Home & Living 236,678,299.13     77.66         B 0.13         Y   
4                Footwear 195,768,282.63     86.66         B 0.13         Y   
5                 Fashion 157,875,250.20     93.92         C 0.12         Y   
6  Beauty & Personal Care 101,623,204.40     98.59         C 0.11         Y   
7      Grocery Essentials  30,584,282.76    100.00         C 0.12         Y   

  ABC_XYZ_Matrix  
0             AY  
1             AY  
2             AX  
3             BY  
4             BY  
5             CY  
6             CY  
7             CY  


=== 2. SAFETY STOCK & ROP (TOP REGIONAL NODES) ===
     zone              